# Trie (Prefix Tree)

*Insert · Search · Autocomplete · Delete · Real-World*



---
## 🧠 Mental Model: Trie (Prefix Tree)

> **A trie is a tree where each path from root to a node spells out a prefix of stored strings.**  
> It trades space (one node per character) for O(L) lookup/insert where L = string length — completely independent of the number of strings stored.

### WHY — Why does it exist?
Hash maps give O(1) *exact* key lookup. Tries give O(L) lookup AND:
- All strings that **start with a prefix** (autocomplete, autocorrect)
- **Longest common prefix** between two strings
- **Wildcard matching** (`.` matches any character)

These prefix queries are O(L) with a trie but O(n·L) by scanning all strings.

### WHAT — Structure

```
Strings: ["cat", "car", "card", "care", "bat"]

root
├── c
│   └── a
│       ├── t  ← "cat" (is_word=True)
│       └── r  ← "car" (is_word=True)
│           ├── d  ← "card" (is_word=True)
│           └── e  ← "care" (is_word=True)
└── b
    └── a
        └── t  ← "bat" (is_word=True)

Each node: {children: dict[char → TrieNode], is_word: bool}
```

### HOW — Core operations

**Insert `"card"`:**  root → c → a → r → d (mark `is_word=True`)

**Search `"car"`:**  root → c → a → r → check `is_word` → True

**Starts-with `"ca"`:**  root → c → a → exists → True (don't need `is_word`)

**Autocomplete `"ca"`:**  DFS from `a` node, collect all words in subtree

### WHEN — Use cases

| Problem | Why trie |
|---------|---------|
| Autocomplete / autocorrect | Prefix search in O(L) |
| IP routing tables (CIDR) | Longest-prefix match |
| Dictionary word validation | O(L) lookup per word |
| Spell checker | Similar-prefix suggestions |
| Word puzzle solving | All words in a grid |
| Phone book prefix search | O(L) independent of n |

**When NOT to use a trie:**
- Simple exact-key lookup → use `dict` (simpler, faster in practice)
- Keys are not strings/sequences → use `dict`
- Memory is tight with long keys → trie uses O(n·L) nodes

```
Complexity:
  Insert:       O(L)   where L = length of the string
  Search:       O(L)
  Starts-with:  O(L)
  Space:        O(n · L · A)  n = words, A = alphabet size
```

**Gotcha** — A standard trie stores one child per character per node (`dict[str, TrieNode]`). For a 26-letter alphabet, that's up to 26 pointers per node. For a large alphabet (Unicode), prefer a compressed trie or radix tree.


In [ ]:

from __future__ import annotations
from collections import defaultdict


class TrieNode:
    def __init__(self) -> None:
        self.children: dict[str, TrieNode] = {}
        self.is_word = False


class Trie:
    """
    Prefix tree: O(L) insert/search/starts_with where L = key length.
    Used for: autocomplete, IP routing, spell-check, word puzzles.
    """

    def __init__(self) -> None:
        self.root = TrieNode()

    def insert(self, word: str) -> None:
        node = self.root
        for ch in word:
            node = node.children.setdefault(ch, TrieNode())
        node.is_word = True

    def search(self, word: str) -> bool:
        """Return True iff the exact word was inserted."""
        node = self._find_prefix_node(word)
        return node is not None and node.is_word

    def starts_with(self, prefix: str) -> bool:
        """Return True if any inserted word starts with prefix."""
        return self._find_prefix_node(prefix) is not None

    def autocomplete(self, prefix: str) -> list[str]:
        """Return all inserted words that start with prefix (sorted)."""
        node = self._find_prefix_node(prefix)
        results: list[str] = []
        if node:
            self._dfs(node, list(prefix), results)
        return sorted(results)

    def _find_prefix_node(self, prefix: str) -> TrieNode | None:
        node = self.root
        for ch in prefix:
            if ch not in node.children:
                return None
            node = node.children[ch]
        return node

    def _dfs(self, node: TrieNode, path: list[str], results: list[str]) -> None:
        if node.is_word:
            results.append("".join(path))
        for ch, child in node.children.items():
            path.append(ch)
            self._dfs(child, path, results)
            path.pop()


# ── Demo ──────────────────────────────────────────────────────────────────
trie = Trie()
for word in ["cat", "car", "card", "care", "bat", "ball", "band"]:
    trie.insert(word)

assert trie.search("cat")
assert trie.search("card")
assert not trie.search("ca")        # prefix only, not a full word
assert trie.starts_with("ca")       # prefix exists
assert not trie.starts_with("xyz")

suggestions = trie.autocomplete("ca")
assert suggestions == ["car", "card", "care", "cat"]
print("Trie autocomplete('ca'):", suggestions)

suggestions_b = trie.autocomplete("ba")
assert suggestions_b == ["ball", "band", "bat"]
print("Trie autocomplete('ba'):", suggestions_b)

print("All trie demos passed ✓")


---
## OPEN ADDRESSING (linear probing): all entries live in ONE array. On collision


In [ ]:
_EMPTY = object()
_DELETED = object()


class OpenAddressingHashMap:
    def __init__(self, capacity: int = 8):
        self._cap = capacity
        self._size = 0
        self._keys = [_EMPTY] * capacity
        self._vals = [None] * capacity

    def _probe(self, key):
        i = hash(key) % self._cap
        first_deleted = None
        for _ in range(self._cap):
            slot = self._keys[i]
            if slot is _EMPTY:
                return (first_deleted if first_deleted is not None else i), False
            if slot is _DELETED:
                if first_deleted is None:
                    first_deleted = i
            elif slot == key:
                return i, True
            i = (i + 1) % self._cap         # linear probe to the next slot
        return (first_deleted if first_deleted is not None else -1), False

    def _resize(self, new_cap: int) -> None:
        items = [(k, v) for k, v in zip(self._keys, self._vals)
                 if k is not _EMPTY and k is not _DELETED]
        self._cap = new_cap
        self._keys = [_EMPTY] * new_cap
        self._vals = [None] * new_cap
        self._size = 0
        for k, v in items:
            self.put(k, v)

    def put(self, key, value) -> None:
        if (self._size + 1) / self._cap > 0.6:   # keep open addressing loose
            self._resize(self._cap * 2)
        i, found = self._probe(key)
        if not found:
            self._size += 1
        self._keys[i] = key
        self._vals[i] = value

    def get(self, key, default=None):
        i, found = self._probe(key)
        return self._vals[i] if found else default

    def delete(self, key) -> bool:
        i, found = self._probe(key)
        if not found:
            return False
        self._keys[i] = _DELETED            # tombstone, NOT empty
        self._vals[i] = None
        self._size -= 1
        return True

    def __len__(self) -> int:
        return self._size


def _exercise(make_map) -> None:
    m = make_map()
    # Insert enough to force multiple resizes.
    for i in range(100):
        m.put(f"key{i}", i)
    assert len(m) == 100
    assert m.get("key42") == 42
    assert m.get("nope", -1) == -1
    m.put("key42", 999)                     # update, not insert
    assert m.get("key42") == 999
    assert len(m) == 100
    assert m.delete("key42") is True
    assert m.get("key42") is None
    assert len(m) == 99
    # Behaves like a real dict for the same operations.
    ref = {f"key{i}": i for i in range(100)}
    ref["key42"] = 999
    del ref["key42"]
    for k in ref:
        assert m.get(k) == ref[k]


def demo() -> None:
    _exercise(lambda: ChainingHashMap(capacity=8))
    print("   chaining: 100 inserts + resizes + update + delete match a real dict")
    _exercise(lambda: OpenAddressingHashMap(capacity=8))
    print("   open addressing: linear probing + tombstones + resize match a real dict")

    # Show a collision explicitly: two keys forced into the same initial slot.
    m = ChainingHashMap(capacity=4)
    m.put("a", 1)
    m.put("e", 2)   # different key; may or may not collide, but both retrievable
    assert m.get("a") == 1 and m.get("e") == 2
    print("   collisions resolved: distinct keys in the same slot are still found correctly")


def main() -> None:
    print("=" * 70)
    print("DSA INTERNALS — hash_tables.py")
    print("=" * 70)
    print("Building dict/set from scratch (the two collision strategies):")
    demo()
    print("-" * 70)
    print("Lesson: O(1) is average+amortized; a bad hash or full table -> O(n). Resize keeps it fast.")
    print("All hash_tables demos passed ✔")


if __name__ == "__main__":
    # Keep Unicode output safe even when stdout is redirected/piped (Windows cp1252 fallback).
    import sys
    if hasattr(sys.stdout, "reconfigure"):
        sys.stdout.reconfigure(encoding="utf-8")
    main()